In [6]:
import os
import shutil
import random
from collections import defaultdict
import pandas as pd

In [7]:
image_folder = "../data/CityInfant/BlackWhite"  
#output_folder = "../data/CityInfant/train-test" 
output_folder = "../data/CityInfant/train-test-2" 
validation_file = "../data/CityInfant/Validationinfo/labels.csv"

In [8]:
df = pd.read_csv(validation_file)
selected_images = set(df["Image"].tolist())

image_files = [f for f in os.listdir(image_folder) if f in selected_images]

# Extract unique baby IDs
baby_dict = defaultdict(list)
for file in image_files:
    baby_id = file.split('Neg')[0] if 'Neg' in file else file.split('Pos')[0] if 'Pos' in file else file.split('Neu')[0]
    baby_dict[baby_id].append(file)

# Get list of unique baby IDs
baby_ids = list(baby_dict.keys())
len(baby_ids)

68

In [ ]:
# Set the number of splits and the ratio
num_splits = 5
split_ratio = 0.8  # 80% train, 20% test

for split_idx in range(1, num_splits + 1):
    random.shuffle(baby_ids)  # Shuffle each time for randomness
    split_point = int(len(baby_ids) * split_ratio)
    
    train_babies = set(baby_ids[:split_point])
    test_babies = set(baby_ids[split_point:])
    
    train_folder = os.path.join(output_folder, f"train{split_idx}")
    test_folder = os.path.join(output_folder, f"test{split_idx}")
    os.makedirs(train_folder, exist_ok=True)
    os.makedirs(test_folder, exist_ok=True)
    
    # Move only the selected 151 images to train/test folders
    train_count, test_count = 0, 0
    for baby, files in baby_dict.items():
        target_folder = train_folder if baby in train_babies else test_folder
        for file in files:
            if file in selected_images:  # Only move files in labels.csv
                shutil.copy(os.path.join(image_folder, file), os.path.join(target_folder, file))
                if target_folder == train_folder:
                    train_count += 1
                else:
                    test_count += 1

    print(f"Split {split_idx} created: {train_count} images in train, {test_count} images in test")


In [10]:
# Shuffle to randomize order
random.shuffle(baby_ids)

# Divide babies into 5 approximately equal groups (for test sets)
num_splits = 5
split_size = len(baby_ids) // num_splits
test_splits = [set(baby_ids[i * split_size:(i + 1) * split_size]) for i in range(num_splits)]

# Assign remaining babies (if any) to the last group
remaining_babies = set(baby_ids[num_splits * split_size:])
for i in range(len(remaining_babies)):
    test_splits[i % num_splits].add(remaining_babies.pop())

# Create train-test splits
for split_idx in range(1, num_splits + 1):
    test_babies = test_splits[split_idx - 1]
    train_babies = set(baby_ids) - test_babies  # All other babies become training set

    # Create output folders
    train_folder = os.path.join(output_folder, f"train{split_idx}")
    test_folder = os.path.join(output_folder, f"test{split_idx}")
    os.makedirs(train_folder, exist_ok=True)
    os.makedirs(test_folder, exist_ok=True)

    # Move files to respective train/test folders
    train_count, test_count = 0, 0
    for baby, files in baby_dict.items():
        target_folder = train_folder if baby in train_babies else test_folder
        for file in files:
            if file in selected_images:  # Only move files listed in labels.csv
                shutil.copy(os.path.join(image_folder, file), os.path.join(target_folder, file))
                if target_folder == train_folder:
                    train_count += 1
                else:
                    test_count += 1

    print(f"Split {split_idx} created: {train_count} images in train, {test_count} images in test")

Split 1 created: 117 images in train, 32 images in test
Split 2 created: 116 images in train, 33 images in test
Split 3 created: 121 images in train, 28 images in test
Split 4 created: 122 images in train, 27 images in test
Split 5 created: 120 images in train, 29 images in test
